# Train Face Attribute Model

In [1]:
import json
from collections import Counter
from pathlib import Path
import random
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

def format_seconds(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f'{hours:d}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'

def resolve_training_device(torch_module, require_gpu: bool = True) -> str:
    if torch_module.cuda.is_available():
        return 'cuda'
    if require_gpu:
        raise RuntimeError(
            'CUDA GPU is required for this training run, but the current PyTorch build does not have CUDA available. '
            'Install a CUDA-enabled PyTorch build into .venv and restart the notebook kernel.'
        )
    return 'cpu'

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

from systems.static_auto_tryon.auto_app.ml.datasets import MultiAttributeDataset, read_jsonl_manifest
from systems.static_auto_tryon.auto_app.ml.face_training_utils import FaceTrainImageTransform
from systems.static_auto_tryon.auto_app.ml.hairstyle_attribute_model import build_attribute_model, multitask_cross_entropy
from systems.static_auto_tryon.auto_app.ml.transforms import ResizeImage

PROJECT_ROOT


WindowsPath('.')

In [ ]:
DATASET_ROOT = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'celeba_face_strong'
TRAIN_MANIFEST = DATASET_ROOT / 'train.jsonl'
VAL_MANIFEST = DATASET_ROOT / 'val.jsonl'
VOCAB_PATH = DATASET_ROOT / 'label_vocab.json'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'

LIGHTWEIGHT_MODE = False
REQUIRE_GPU = True
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'models' / 'celeba_face_strong' / ('face_attribute_model_light.pt' if LIGHTWEIGHT_MODE else 'face_attribute_model.pt')

FACE_FIELDS = ('gender', 'face_fullness', 'cheekbones', 'hairline')
SEED = 42
ARCHITECTURE = 'resnet34'
IMAGE_SIZE = 160 if LIGHTWEIGHT_MODE else 224
BATCH_SIZE = 32 if LIGHTWEIGHT_MODE else 96
EPOCHS = 3 if LIGHTWEIGHT_MODE else 12
LEARNING_RATE = 1e-3 if LIGHTWEIGHT_MODE else 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0 if LIGHTWEIGHT_MODE else 4
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0 
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None
USE_AMP = True
USE_TORCH_COMPILE = False
PATIENCE = 3 if LIGHTWEIGHT_MODE else 5
DEVICE = resolve_training_device(torch, require_gpu=REQUIRE_GPU)


In [3]:
train_records = read_jsonl_manifest(TRAIN_MANIFEST)
val_records = read_jsonl_manifest(VAL_MANIFEST)
label_vocab = json.loads(VOCAB_PATH.read_text(encoding='utf-8'))
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

pd.Series(
    {
        'lightweight_mode': LIGHTWEIGHT_MODE,
        'train_records': len(train_records),
        'val_records': len(val_records),
        'device': DEVICE,
        'torch_version': torch.__version__,
        'cuda_version': torch.version.cuda,
        'architecture': ARCHITECTURE,
    }
)


lightweight_mode           False
train_records             151039
val_records                18517
device                      cuda
torch_version       2.11.0+cu128
cuda_version                12.8
architecture            resnet34
dtype: object

In [4]:
def build_sample_weights(records, fields):
    field_counts = {field: Counter(str(record['labels'][field]) for record in records) for field in fields}
    weights = []
    for record in records:
        score = 0.0
        for field in fields:
            score += 1.0 / max(field_counts[field][str(record['labels'][field])], 1)
        weights.append(score / max(len(fields), 1))
    return weights

def build_class_weights(records, fields, label_vocab, device):
    output = {}
    for field in fields:
        counts = Counter(str(record['labels'][field]) for record in records)
        weights = torch.ones(len(label_vocab[field]), dtype=torch.float32, device=device)
        total = sum(counts.values())
        for label, index in label_vocab[field].items():
            count = counts.get(label, 0)
            if count > 0:
                weights[index] = total / count
        output[field] = weights / weights.mean().clamp_min(1e-6)
    return output

set_seed(SEED)
train_transform = FaceTrainImageTransform((IMAGE_SIZE, IMAGE_SIZE))
val_transform = ResizeImage((IMAGE_SIZE, IMAGE_SIZE))
train_dataset = MultiAttributeDataset(train_records, label_vocab=label_vocab, transform=train_transform, fields=FACE_FIELDS)
val_dataset = MultiAttributeDataset(val_records, label_vocab=label_vocab, transform=val_transform, fields=FACE_FIELDS)

sample_weights = build_sample_weights(train_records, FACE_FIELDS)
loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
}
if NUM_WORKERS > 0:
    loader_kwargs['persistent_workers'] = PERSISTENT_WORKERS
    loader_kwargs['prefetch_factor'] = PREFETCH_FACTOR

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True),
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    **loader_kwargs,
)

model = build_attribute_model(
    attribute_vocab_sizes={field: len(label_vocab[field]) for field in FACE_FIELDS},
    base_channels=32,
    dropout=0.3,
    architecture=ARCHITECTURE,
).to(DEVICE)
if DEVICE == 'cuda':
    model = model.to(memory_format=torch.channels_last)
if USE_TORCH_COMPILE and hasattr(torch, 'compile'):
    model = torch.compile(model)
scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE == 'cuda'))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
class_weights = build_class_weights(train_records, FACE_FIELDS, label_vocab, DEVICE)

sum(parameter.numel() for parameter in model.parameters())


21288776

In [5]:
def move_targets_to_device(targets, device):
    return {field: tensor.to(device, non_blocking=True) for field, tensor in targets.items()}

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        if device == 'cuda':
            images = images.to(memory_format=torch.channels_last)
        targets = move_targets_to_device(batch['labels'], device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=(USE_AMP and device == 'cuda')):
            outputs = model(images)
            loss = multitask_cross_entropy(outputs, targets, class_weights=class_weights, label_smoothing=0.03)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)

@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    field_correct = {field: 0 for field in FACE_FIELDS}
    exact_correct = 0
    total_samples = 0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        if device == 'cuda':
            images = images.to(memory_format=torch.channels_last)
        targets = move_targets_to_device(batch['labels'], device)
        with torch.amp.autocast(device_type='cuda', enabled=(USE_AMP and device == 'cuda')):
            outputs = model(images)
            loss = multitask_cross_entropy(outputs, targets, class_weights=class_weights, label_smoothing=0.0)
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size
        batch_matches = []
        for field in FACE_FIELDS:
            predictions = outputs[field].argmax(dim=1)
            matches = predictions == targets[field]
            field_correct[field] += int(matches.sum().item())
            batch_matches.append(matches)
        exact_correct += int(torch.stack(batch_matches, dim=0).all(dim=0).sum().item())

    return {
        'val_loss': running_loss / max(total_samples, 1),
        'exact_match_accuracy': exact_correct / max(total_samples, 1),
        **{field: field_correct[field] / max(total_samples, 1) for field in FACE_FIELDS},
    }


In [6]:
history = []
best_metric = float('-inf')
best_epoch = None
epoch_durations = []
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    mode_label = 'lightweight' if LIGHTWEIGHT_MODE else 'full'
    print(f'Running epoch {epoch}/{EPOCHS} [{mode_label}]...')

    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    metrics = evaluate(model, val_loader, DEVICE)
    scheduler.step(metrics['val_loss'])

    epoch_seconds = time.perf_counter() - epoch_start
    epoch_durations.append(epoch_seconds)
    current_metric = metrics['exact_match_accuracy']
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    average_epoch_seconds = sum(epoch_durations) / len(epoch_durations)
    remaining_seconds = average_epoch_seconds * (EPOCHS - epoch)

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        **metrics,
        'learning_rate': float(optimizer.param_groups[0]['lr']),
        'epoch_time': format_seconds(epoch_seconds),
        'best_epoch_so_far': best_epoch,
        'best_exact_match_so_far': round(best_metric, 4),
    }
    history.append(row)

    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'model_state_dict': model.state_dict(),
            'history': history,
            'last_epoch': epoch,
            'best_epoch': best_epoch,
            'best_exact_match': best_metric,
            'core_fields': FACE_FIELDS,
            'label_vocab': label_vocab,
            'architecture': ARCHITECTURE,
            'base_channels': 32,
            'dropout': 0.3,
            'resolution': IMAGE_SIZE,
            'train_count': len(train_records),
            'val_count': len(val_records),
        },
        CHECKPOINT_PATH,
    )

    print(f'Completed epoch {epoch}/{EPOCHS}')
    print(f"Epoch time: {format_seconds(epoch_seconds)}")
    print(f"Estimated remaining: {format_seconds(remaining_seconds)}")
    print(row)

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping triggered after {epoch} epochs.')
        break

print(f'Saved checkpoint to {CHECKPOINT_PATH}')
pd.DataFrame(history)


Running epoch 1/12 [full]...
Completed epoch 1/12
Epoch time: 27:22
Estimated remaining: 5:01:04
{'epoch': 1, 'train_loss': 1.2974028307116168, 'val_loss': 1.6803980706068586, 'exact_match_accuracy': 0.34751849651671435, 'gender': 0.9018739536642004, 'face_fullness': 0.6852081870713399, 'cheekbones': 0.8546740832748285, 'hairline': 0.556893665280553, 'learning_rate': 0.0003, 'epoch_time': '27:22', 'best_epoch_so_far': 1, 'best_exact_match_so_far': 0.3475}
Running epoch 2/12 [full]...
Completed epoch 2/12
Epoch time: 48:59
Estimated remaining: 6:21:50
{'epoch': 2, 'train_loss': 1.0790386187833327, 'val_loss': 1.2575683741369432, 'exact_match_accuracy': 0.5226008532699682, 'gender': 0.9798563482205541, 'face_fullness': 0.7235513312091592, 'cheekbones': 0.8694172922179618, 'hairline': 0.7738294540152293, 'learning_rate': 0.0003, 'epoch_time': '48:59', 'best_epoch_so_far': 2, 'best_exact_match_so_far': 0.5226}
Running epoch 3/12 [full]...
Completed epoch 3/12
Epoch time: 25:34
Estimated re

,epoch,train_loss,val_loss,exact_match_accuracy,gender,face_fullness,cheekbones,hairline,learning_rate,epoch_time,best_epoch_so_far,best_exact_match_so_far
0,1,1.297403,1.680398,0.347518,0.901874,0.685208,0.854674,0.556894,0.000300,27:22,1,0.3475
1,2,1.079039,1.257568,0.522601,0.979856,0.723551,0.869417,0.773829,0.000300,48:59,2,0.5226
2,3,0.999843,1.123738,0.662040,0.976562,0.884431,0.872280,0.849058,0.000300,25:34,3,0.6620
3,4,0.932891,1.114407,0.677810,0.978290,0.897554,0.877464,0.854080,0.000300,25:33,4,0.6778
4,5,0.876372,1.107756,0.671815,0.983529,0.887077,0.877032,0.846681,0.000300,25:41,4,0.6778
5,6,0.821664,1.121454,0.692877,0.984771,0.910245,0.874818,0.863747,0.000300,27:24,6,0.6929
6,7,0.778885,1.187281,0.677702,0.982449,0.882810,0.873468,0.875088,0.000300,28:14,6,0.6929
7,8,0.747477,1.219140,0.667063,0.980936,0.879894,0.869741,0.870173,0.000150,28:12,6,0.6929
8,9,0.665483,1.257076,0.747853,0.984987,0.946104,0.876168,0.902522,0.000150,27:58,9,0.7479
9,10,0.633104,1.262008,0.735540,0.986067,0.939029,0.869849,0.899012,0.000150,28:22,9,0.7479


In [7]:
payload = torch.load(CHECKPOINT_PATH, map_location='cpu')
pd.Series(
    {
        'checkpoint_path': str(CHECKPOINT_PATH),
        'best_epoch': payload.get('best_epoch'),
        'best_exact_match': payload.get('best_exact_match'),
        'train_count': payload.get('train_count'),
        'val_count': payload.get('val_count'),
        'architecture': payload.get('architecture'),
    }
)


checkpoint_path     PROJECT_ROOT/...
best_epoch                                                         12
best_exact_match                                             0.764973
train_count                                                    151039
val_count                                                       18517
architecture                                                 resnet34
dtype: object